## PydanticAI 이미지 이해

- [PydanticAI 이미지/오디오/비디오/문서 입력 공식 문서](https://ai.pydantic.dev/input/#image-input)
- [Gemini API 이미지 이해 문서](https://ai.google.dev/gemini-api/docs/vision)

**실습 목표**
- PydanticAI의 `BinaryContent`, `ImageUrl`을 사용하여 이미지를 Agent에 전달하는 방법을 익힙니다.
- 단일/다중 이미지 분석, 객체 감지(구조화된 출력)를 실습합니다.

### Gemini 이미지 이해 기초

#### 지원 모델
| 모델 시리즈 | 버전 | 특징 |
|------------|------|------|
| **Gemini 2.5** | Flash, Flash-Lite, Pro | 최신 기능, 세분화 지원 |
| **Gemini 2.0** | Flash, Flash-Lite | 객체 감지 강화 |

**처리 한도** : 요청당 최대 **3,600개** 이미지


#### 지원 이미지 형식
| 형식 | MIME 타입 | 특징 |
|:----:|:---------:|:-----|
| PNG | `image/png` | 무손실, 투명도  |
| JPEG | `image/jpeg` | 범용, 압축 |
| WEBP | `image/webp` | Google 개발, 효율적 |
| HEIC/HEIF | `image/heic`, `image/heif` | Apple 기본 형식 |

#### 고급 기능
| 기능 | 지원 모델 | 주요 특징 |
|------|----------|:-----------|
| **객체 감지** | Gemini 2.0+ | • 경계 상자 [ymin, xmin, ymax, xmax]<br>• 0-1000 정규화 좌표<br>• 맞춤 라벨 지원 |
| **세그멘테이션** | Gemini 2.5+ | • JSON 출력 (bbox + mask)<br>• Base64 PNG 마스크<br>|

- 객체 감지/세그멘테이션 같은 시각적 작업에서는 추론 과정이 오히려 정확도를 떨어뜨릴 수 있어서, `thinking_budget=0`으로 설정하여 추론을 비활성화하면 더 나은 결과를 얻을 수 있다고 공식 문서에서 권장하고 있습니다
- thinking_budget은 Gemini 모델이 답변하기 전에 **내부적으로 생각하는 시간(토큰 수)** 을 제어하는 파라미터입니다.

#### 팁 및 권장사항
- 이미지가 올바르게 회전되었는지 확인합니다.
- 흐릿하지 않고 선명한 이미지를 사용하세요.
- 텍스트가 포함된 단일 이미지를 사용하는 경우 contents 배열의 이미지 부분 뒤에 텍스트 프롬프트를 배치합니다.

- 객체 탐지 및 세그멘테이션 예시

![image.png](https://ai.google.dev/static/gemini-api/docs/images/segmentation.jpg?hl=ko)

### 환경 설정

실습에 필요한 패키지를 설치하고, API 키를 설정합니다.

In [ ]:
# 필요한 패키지를 한 번에 설치합니다
# !pip install -q pydantic-ai-slim[google] python-dotenv pydantic Pillow matplotlib

# uv 권장! uv 환경에서는 위 pip 대신 터미널에서 uv sync 명령어로 설치하세요:
# uv sync

**실습**
- 환경변수를 로드하고 API 키가 올바르게 설정되었는지 확인합니다.
- `.env` 파일에 `GEMINI_API_KEY`를 설정한 후 아래 셀을 실행합니다.

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import time
import json
from pprint import pprint
from pathlib import Path
from typing import List

from dotenv import load_dotenv
from PIL import Image
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.models.google import GoogleModelSettings

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')

# PydanticAI는 GEMINI_API_KEY 환경변수를 자동으로 인식합니다
# 모델 ID 형식: 'google-gla:{모델명}'
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

# API 호출 간격 (초)
API_DELAY = 3


### 1. 단일 이미지 분석

PydanticAI에서 이미지를 전달하는 방법은 2가지입니다.

| 방식 | 클래스 | 용도 |
|------|--------|------|
| URL 참조 | `ImageUrl(url=...)` | 웹에 공개된 이미지 URL을 직접 전달 |
| 바이너리 데이터 | `BinaryContent(data=..., media_type=...)` | 로컬 파일, API 응답 등 바이트 데이터 전달 |

**ImageUrl 방식** (URL로 전달)

```python
from pydantic_ai import Agent, ImageUrl

agent = Agent(model_id)
result = await agent.run([
    "이 이미지에 무엇이 있나요?",
    ImageUrl(url="https://example.com/image.png"),
])
```

**BinaryContent 방식** (로컬 파일 전달)

```python
from pydantic_ai import Agent, BinaryContent

# 이미지 파일을 바이트로 읽기
with open("image.jpg", "rb") as f:
    image_bytes = f.read()

# BinaryContent로 감싸서 Agent에 전달
result = await agent.run([
    BinaryContent(data=image_bytes, media_type="image/jpeg"),
    "이 이미지에 무엇이 있나요?",
])
```

| 기존 (google-genai) | PydanticAI |
|---|---|
| `types.Part.from_bytes(data, mime_type)` | `BinaryContent(data, media_type)` |
| `client.models.generate_content(model, contents)` | `await agent.run([image, question])` |
| `response.text` | `result.output` |

**실습**
- 이미지를 읽고 PydanticAI Agent로 분석합니다.
- `BinaryContent`로 이미지를 감싸고, `await agent.run()`으로 질문합니다.

In [ ]:
# ==========================================
# 단일 이미지 분석
# ==========================================

def load_image_content(image_path):
    """이미지 파일을 BinaryContent로 변환하는 헬퍼 함수"""
    path = Path(image_path)
    
    # 파일 확장자로 media_type 결정
    suffix = path.suffix.lower()
    media_types = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.webp': 'image/webp',
        '.heic': 'image/heic',
        '.heif': 'image/heif'
    }
    media_type = media_types.get(suffix, 'image/jpeg')
    
    # 이미지 바이트 읽기
    with open(path, 'rb') as f:
        image_bytes = f.read()
    
    # BinaryContent로 변환
    return BinaryContent(data=image_bytes, media_type=media_type)


# Agent 생성
image_agent = Agent(model_id)

# 이미지 로드
image_path = "./data/sample_images/실습 이미지1.jpeg"
image_content = load_image_content(image_path)

# 다양한 질문으로 이미지 분석
questions = [
    "이 이미지에 무엇이 있나요?",
    "이미지의 색상과 분위기를 설명해주세요.",
    "이미지에서 주목할 만한 특징을 3가지 말해주세요."
]

print("=== 이미지 분석 결과 ===")
print()

for i, question in enumerate(questions, 1):
    print(f"[질문 {i}] {question}")
    
    # PydanticAI Agent 실행 — 이미지 + 질문을 리스트로 전달
    result = await image_agent.run([image_content, question])
    
    print(f"[답변] {result.output}")
    print("-" * 50)

### 2. 다중 이미지 비교 분석

여러 이미지를 동시에 전달하려면 **BinaryContent** 를 여러 개 리스트에 포함하면 됩니다.

```python
result = await agent.run([image1, image2, "두 이미지를 비교해주세요."])
```

**실습**
- 두 장의 광고 이미지를 비교 분석합니다.
- 여러 `BinaryContent`를 리스트에 담아 Agent에 전달합니다. `GoogleModelSettings`로 매개변수를 제어합니다.

In [ ]:
# ==========================================
# 다중 이미지 비교 분석
# ==========================================

# 이미지 파일 경로
image_paths = [
    "./data/sample_images/광고1.jpg",
    "./data/sample_images/광고2.jpg",
]

# 여러 이미지를 BinaryContent로 로드
image_contents = []
for path in image_paths:
    image_contents.append(load_image_content(path))
    print(f"  - {Path(path).name} 로드 완료")

# 다양한 관점의 비교 분석 질문들
comparison_questions = {
    "비즈니스 관점": """
마케팅 전문가의 관점에서 두 광고 이미지를 심층 비교 분석해주세요:

## 1. 광고 효과성 직접 비교
- 각 광고의 핵심 메시지 전달력 (1-10점 평가 및 이유)
- 시선 집중도와 주목도 비교
- 브랜드 인지도 향상 기여도
- 구매 전환율 예상 (어떤 광고가 더 높을지 근거와 함께)

## 2. 타겟 고객 세그먼트별 효과
- 연령대별 선호도 예측 
- 성별 반응 차이 분석
- 각 광고가 가장 효과적일 타겟 페르소나 정의

## 3. 채널별 활용 전략
광고1과 광고2 중 어느 것이 더 효과적인지 채널별로 평가:
- 인스타그램 피드/스토리
- 옥외광고(버스/지하철)
- 온라인 배너 광고
- TV/유튜브 광고

## 4. 최종 추천
- 현재 시점에서 집중 투자해야 할 광고는? (명확한 1개 선택 및 이유)
- 두 광고의 최적 활용 시나리오 (동시 활용 vs 단계별 활용)
- 즉시 실행 가능한 개선 방안 3가지 (우선순위별)
""",
    "창의적 관점": """
이미지를 창의적으로 해석해주세요:
- 각 이미지를 한 문장의 시로 표현한다면?
- 이미지들이 연결된 스토리를 만든다면 어떤 이야기일까요?
- 각 이미지에 어울리는 음악 장르와 그 이유
- 이미지를 합쳐 새로운 작품을 만든다면 어떤 컨셉이 좋을까요?
"""
}

# Agent 생성
compare_agent = Agent(model_id)

# 매개변수 설정
settings = GoogleModelSettings(
    temperature=0.5,
    # max_tokens=100,
)

print("=== 다중 이미지 비교 분석 결과 ===")
print()
print(f"비교할 이미지: {len(image_paths)}개")
for path in image_paths:
    print(f"  - {Path(path).name}")
print("=" * 80)

# 각 관점별로 분석 수행
for title, question in comparison_questions.items():
    print(f"{title}")
    print("-" * 40)
    print(f"[질문]")
    print(question)
    
    # 이미지들 + 질문을 리스트로 전달
    contents = image_contents + [question]
    result = await compare_agent.run(contents, model_settings=settings)
    
    print(f"[답변]")
    print(result.output)
    print("=" * 80)
    
    # API 호출 간격 대기
    time.sleep(API_DELAY)

### 3. 객체 감지 (Object Detection)

PydanticAI의 **구조화된 출력(output_type)** 을 활용하면, 객체 감지 결과를 Pydantic 모델로 직접 받을 수 있습니다.

| 기존 방식 | PydanticAI 방식 |
|---|---|
| `response_mime_type='application/json'` | `Agent(output_type=DetectionResult)` |
| `json.loads(response.text)` 수동 파싱 | `result.output` 자동 파싱 |
| 파싱 오류 직접 처리 | Pydantic 검증 자동 적용 |

**실습**
- Pydantic 스키마를 정의하고, Agent로 객체를 감지합니다.
- `output_type`으로 구조화된 출력을 강제하여, JSON 파싱 없이 바로 Python 객체로 받습니다.

In [ ]:
# ==========================================
# 객체 감지 (Object Detection) - 구조화된 출력 활용
# ==========================================

# 객체 감지 결과 스키마 정의
class DetectedObject(BaseModel):
    """감지된 개별 객체 정보"""
    label: str = Field(description="객체 이름")
    box_2d: List[int] = Field(
        description="경계 상자 좌표 [ymin, xmin, ymax, xmax], 0-1000 정규화",
        min_length=4,
        max_length=4
    )

class DetectionResult(BaseModel):
    """이미지 객체 감지 전체 결과"""
    objects: List[DetectedObject] = Field(description="감지된 객체 목록")

# 객체 감지 Agent 생성 — output_type으로 구조화된 출력 강제
detect_agent = Agent(
    model_id,
    output_type=DetectionResult,
    system_prompt="""이미지의 모든 주요 객체를 감지하세요.
각 객체에 대해 label(객체 이름)과 box_2d([ymin, xmin, ymax, xmax]) 좌표를 제공하세요.
box_2d는 0-1000으로 정규화된 좌표입니다.""",
)

# 이미지 로드
image_path = "./data/sample_images/식재료.jpg"
image_content = load_image_content(image_path)

# PIL로 이미지 크기 확인 (좌표 변환에 필요)
pil_image = Image.open(image_path)
width, height = pil_image.size

# Agent 실행 — 이미지 전달
result = await detect_agent.run(
    [image_content, "이미지의 모든 주요 객체를 감지하세요."]
)

# result.output => DetectionResult 객체 (자동 파싱)
detection = result.output

# 정규화 좌표를 실제 픽셀 좌표로 변환
converted_boxes = []
for obj in detection.objects:
    box = obj.box_2d
    abs_y1 = int(box[0] / 1000 * height)
    abs_x1 = int(box[1] / 1000 * width)
    abs_y2 = int(box[2] / 1000 * height)
    abs_x2 = int(box[3] / 1000 * width)
    
    converted_boxes.append({
        "label": obj.label,
        "box": [abs_x1, abs_y1, abs_x2, abs_y2]
    })

# 결과 출력
print("=== 객체 감지 결과 ===")
print()
print(f"이미지 크기: ({width}, {height})")
print(f"감지된 객체 수: {len(detection.objects)}개")
print()

for obj in converted_boxes:
    print(f"객체: {obj['label']}")
    print(f"좌표: {obj['box']}")
    print("-" * 30)

# 시각화를 위해 결과를 딕셔너리로 저장
detection_result = {
    "image_size": (width, height),
    "objects": converted_boxes
}

객체 탐지 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def draw_bounding_boxes(image_path, detection_result, output_path="result.jpg"):
    """감지된 객체에 바운딩 박스와 라벨을 그려서 저장"""
    
    # 이미지 열기
    image = Image.open(image_path)
    
    # 그림 생성
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)
    
    # 각 객체 그리기
    for obj in detection_result['objects']:
        box = obj['box']  # [x1, y1, x2, y2]
        label = obj['label']
        
        # 바운딩 박스 그리기
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        
        # 라벨 텍스트 그리기
        ax.text(x1, y1-5, label, 
                bbox=dict(facecolor='red', alpha=0.7),
                color='white', fontsize=10)
    
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"결과 이미지 저장 완료: {output_path}")

# 실행
draw_bounding_boxes(image_path, detection_result, "data/sample_images/detected_objects.jpg")